# 차량 5종 객체 탐지 실습 — 데이터 수집·CVAT 라벨링부터 NPU 시연까지
> **YOLOv9-DPI + BlackSwan NPU (Tachy-Shield) 실습 노트북 (CVAT 라벨링 확장판)**
>
> 차량 5종(**ambulance, bus, car, motorcycle, truck**)을 탐지하는 모델을 만듭니다.
> 이번 실습의 확장 포인트는 **데이터를 직접 다루는 경험**입니다: 공개 데이터셋(Open Images)에서 원천 이미지를 수집하고,
> 그중 일부를 **CVAT에서 직접 라벨링**한 뒤, 내 라벨의 품질을 **정량 채점**하고, 학습에 반영합니다.

## 전체 흐름

```
[Colab/T4]  환경 구성 → 데이터 수집(Open Images) → CVAT 라벨링 체험 40장 → 라벨 채점·병합
          → 1단계 학습 → 2단계 학습 → TACHY 컴파일 → 시연용 슬라이드 생성
[CVAT]      프로젝트 생성 → 라벨 등록 → 태스크 업로드 → 라벨링 → YOLO 1.1 Export
[RPi4+NPU]  .tachyrt 복사 → 설정 수정(5클래스) → 실시간 추론 → 모니터 재촬영 시연
```

> 🔑 **핵심 메시지**: *"모델 성능의 상한은 라벨 품질이 정한다."*
> 지금까지는 남이 라벨링한 데이터를 썼습니다. 오늘은 내가 라벨러가 되어보고,
> 내 라벨을 참조 라벨과 **IoU로 대조 채점**해서 "라벨 노이즈가 어디서 생기는지"를 몸으로 확인합니다.

## 예상 소요 시간 (T4 기준)

| 단계 | 소요 시간 |
| --- | --- |
| 환경 구성 (conda + 패키지) | 약 10~15분 |
| 데이터 수집 (Open Images 800장) | 약 10~15분 |
| **CVAT 라벨링 체험 (40장)** | **약 30~40분** (수업 진행상 병렬 가능 — 3-3 참고) |
| 라벨 채점·병합 | 약 5분 |
| 1단계 학습 (bsnet-t, 20 epochs) | 약 25~35분 |
| 2단계 학습 (bsnet-t-o, 20 epochs) | 약 25~35분 |
| TACHY 컴파일 | 약 5~10분 |
| **합계** | **약 2시간~2시간 30분** |

⚠️ **시작 전 체크리스트**
- 런타임 유형이 **GPU (T4)** 인지 확인: `런타임 → 런타임 유형 변경 → T4 GPU`
- `fc.out`, `block_4bit.out` 파일을 **내 PC에** 다운로드 완료 (yolov9-dpi README 안내 경로 — 6장에서 업로드)
- **CVAT 계정** 준비: https://app.cvat.ai 무료 가입 (수업 전날 권장 — 이메일 인증 필요)

⚠️ **이 노트북은 Google Drive를 사용하지 않습니다.** 모든 작업이 Colab 로컬 디스크(`/content`)에서 진행되므로,
**세션이 초기화되면 학습 결과가 사라집니다.** 학습/컴파일이 끝나면 반드시 8-2장의 **산출물 백업 다운로드** 셀을 실행하세요.


---
## 1. Colab 런타임 준비 (Miniforge 직접 설치)

yolov9-dpi는 특정 라이브러리 버전(Python 3.8, torch 1.13.1)만 지원하므로 conda 환경을 사용합니다.

💡 condacolab은 Colab 기본 Python 버전에 고정되어 있어, Colab이 Python 버전을 올리면
`Colab's Python (3.13) does not match expected version` 오류로 깨집니다.
여기서는 **Miniforge를 `/opt/conda`에 직접 설치**하는 방식을 사용합니다.
- Colab Python 버전과 무관하게 동작 (향후 버전 업에도 안전)
- **런타임 재시작 불필요** — 설치 후 바로 다음 셀 진행

설치에 약 1~2분 소요됩니다.

In [ ]:
import os

if not os.path.exists('/opt/conda/bin/conda'):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    !bash /tmp/miniforge.sh -b -p /opt/conda
    !rm /tmp/miniforge.sh
    print('Miniforge 설치 완료')
else:
    print('Miniforge 이미 설치됨')

# 이후 모든 ! 셸 명령에서 conda/mamba를 찾을 수 있도록 PATH 등록
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

설치 확인 — conda와 mamba 버전이 출력되면 정상입니다.

In [ ]:
!conda --version && mamba --version

---
## 2. 저장소 준비 (로컬 디스크)

모든 작업은 Colab 로컬 디스크 `/content` 아래에서 진행합니다. Drive 마운트 과정이 없어 시작이 빠르고 권한 팝업도 없습니다.

⚠️ 대신 세션 초기화 시 결과물이 사라지므로, **8-2장의 백업 다운로드**를 잊지 마세요.

In [ ]:
%cd /content

YOLOv9-DPI 저장소를 복제합니다. (이미 복제되어 있다면 `already exists` 메시지가 나와도 무방합니다.)

In [ ]:
!git clone https://github.com/Deeper-I/yolov9-dpi.git
%cd /content/yolov9-dpi

`py38` 이름의 Python 3.8 conda 환경을 생성합니다.

In [ ]:
!mamba create -n py38 python=3.8 -y

### 2-1. 라이브러리 경로 설정 (ABI 충돌 방지)

conda-forge 최신 빌드의 ICU 라이브러리는 시스템 `libstdc++`보다 새로운 C++ ABI(`CXXABI_1.3.15`)를 요구합니다.
torch가 시스템 libstdc++를 먼저 로드해버리면 이후 `import sqlite3`에서
`` version `CXXABI_1.3.15' not found `` 오류가 발생합니다.

conda 환경 내부의 최신 libstdc++가 우선 로드되도록 `LD_LIBRARY_PATH`를 설정해 해결합니다.
(새 libstdc++는 하위 호환이므로 torch/CUDA에도 안전합니다.)

In [ ]:
import os
os.environ['LD_LIBRARY_PATH'] = '/opt/conda/envs/py38/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])

In [ ]:
%%writefile requirements.txt
torch==1.13.1
torchvision==0.14.1
PyYAML==6.0.1
opencv-python
pandas==1.1.5
DDesignerAPI
IPython
psutil
matplotlib>=3.2.2
seaborn
tensorboard
Pillow==9.4.0
commentjson
ordered_set
onnx==1.14.1
networkx==2.6.3
easydict
tqdm
# compile_linux.py는 0.1.0의 모듈 구조(platform_converter/utils/yolov9)를 기준으로 작성됨
# 0.1.1부터 모듈 재편으로 ModuleNotFoundError 발생 → 반드시 버전 고정
TACHY-Compiler==0.1.0

In [ ]:
!conda run -n py38 pip install -r requirements.txt

### 2-2. 환경 검증

학습 시작 전에 핵심 모듈이 정상 import되는지, GPU가 잡히는지 한 번에 확인합니다.
`OK | torch 1.13.1 ... CUDA: True` 가 나와야 합니다.
(`can not import tensorflow` 경고가 보여도 무시하세요 — 선택적 의존성입니다.)

In [ ]:
!conda run -n py38 python3 -c "import torch, sqlite3, cv2, onnx, tachy_compiler.platform_converter.utils.yolov9.deploy_pt_yolov9; print('OK | torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

---
## 3. 차량 데이터셋 수집 — Open Images V7

Google **Open Images V7**에서 차량 5클래스가 포함된 이미지를 fiftyone으로 수집합니다.
계정·API 키가 필요 없는 공개 경로라 실습 진행이 가볍습니다.

| 항목 | 내용 |
| --- | --- |
| 클래스 | 5개 — `ambulance`(0), `bus`(1), `car`(2), `motorcycle`(3), `truck`(4) *(알파벳순 = 표준 순서)* |
| 규모 | 800장 수집 → train/val/test = 8:1:1 분할 |
| 포맷 | fiftyone이 YOLO 포맷으로 export (변환 코드 불필요) |

💡 African Wildlife와의 차이: 그쪽은 **완제품 데이터셋**(라벨 포함, 균형 분포)이었지만, 이번엔 **현실 데이터**입니다 —
Open Images에서 `car`는 흔하고 `ambulance`는 희귀해서 **심한 클래스 불균형**이 함께 딸려옵니다.
Aquarium 실습의 25배 불균형(fish vs starfish)과 같은 문제가 여기서도 재현되는지 3-1에서 확인하세요.

⚠️ 이 셀은 외부 다운로드라 네트워크 상황에 따라 5~15분 걸립니다. **다운로드 동안 3-3(CVAT 절차)을 먼저 읽어두세요.**

In [ ]:
# fiftyone은 Colab 기본 파이썬에 설치 (py38 환경과 무관 — 데이터 준비 전용 도구)
!pip install -q fiftyone

import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

CANONICAL = ['ambulance', 'bus', 'car', 'motorcycle', 'truck']   # 표준 클래스 순서 (알파벳)
OI_CLASSES = ['Ambulance', 'Bus', 'Car', 'Motorcycle', 'Truck']  # Open Images 표기

ds = foz.load_zoo_dataset(
    'open-images-v7', split='train',
    label_types=['detections'], classes=OI_CLASSES,
    max_samples=800, shuffle=True, seed=42,
    dataset_name='vehicles5',
)
# 5클래스 외 라벨 제거 + 군집 박스(IsGroupOf: 여러 대를 한 박스로 묶은 라벨) 제외
ds = ds.filter_labels('detections',
                      F('label').is_in(OI_CLASSES) & (F('IsGroupOf') == False))
print(ds)
# 📊 기대: 800 샘플. 'ambulance' 포함 이미지는 수십 장 수준일 수 있음 (희귀 클래스)

In [ ]:
# YOLO 포맷 export → 표준 레이아웃(/content/datasets/vehicles)으로 8:1:1 분할
import os, glob, random, shutil
from pathlib import Path

EXPORT = '/content/oi_export'
ds.export(export_dir=EXPORT, dataset_type=fo.types.YOLOv5Dataset,
          label_field='detections',
          classes=[c.lower() for c in OI_CLASSES])   # 소문자 = 표준 순서와 일치

DATASET_DIR = Path('/content/datasets/vehicles')
for sp in ['train', 'val', 'test']:
    (DATASET_DIR / 'images' / sp).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / sp).mkdir(parents=True, exist_ok=True)

imgs = sorted(glob.glob(f'{EXPORT}/images/**/*.jpg', recursive=True))
random.seed(42); random.shuffle(imgs)
n = len(imgs); n_tr, n_va = int(n*0.8), int(n*0.1)
splits = [('train', imgs[:n_tr]), ('val', imgs[n_tr:n_tr+n_va]), ('test', imgs[n_tr+n_va:])]
for sp, group in splits:
    for ip in group:
        stem = Path(ip).stem
        lp = glob.glob(f'{EXPORT}/labels/**/{stem}.txt', recursive=True)
        shutil.copy(ip, DATASET_DIR / 'images' / sp / f'{stem}.jpg')
        if lp:
            shutil.copy(lp[0], DATASET_DIR / 'labels' / sp / f'{stem}.txt')
        else:   # 라벨 없는 이미지는 빈 txt (배경 이미지로 활용)
            (DATASET_DIR / 'labels' / sp / f'{stem}.txt').touch()
    print(f'{sp:>5}: {len(group)}장')

# 클래스 분포 확인 — 불균형의 실체를 숫자로
from collections import Counter
cnt = Counter()
for lp in (DATASET_DIR / 'labels' / 'train').glob('*.txt'):
    for line in lp.read_text().splitlines():
        if line.strip():
            cnt[CANONICAL[int(line.split()[0])]] += 1
print('\ntrain 박스 분포:', dict(cnt))
imb = max(cnt.values()) / max(min(cnt.values()), 1)
print(f'최대/최소 불균형: {imb:.0f}배   # 📊 car 편중 예상 — Aquarium 25배와 비교해볼 것')

### 3-1. CVAT 라벨링 체험 세트 만들기 (40장)

train에서 40장을 뽑아 **라벨을 감춘 zip**(`cvat_upload.zip`)을 만듭니다. 이 40장의 원본 라벨은
`ref_labels/`에 따로 보관합니다 — 여러분이 CVAT에서 단 라벨을 3-5에서 이 **참조 라벨과 대조 채점**하기 위해서입니다.

💡 왜 40장? 라벨링 속도는 초심자 기준 **분당 1~2장**입니다. 40장이면 약 30분 — 라벨링의 고단함을 체감하기엔 충분하고
수업을 잡아먹지는 않는 분량입니다. ("데이터 1만 장 라벨링 외주 견적이 왜 그 가격인지"가 이 30분 뒤에 이해됩니다.)

In [ ]:
import zipfile
CVAT_N = 40
cvat_dir = Path('/content/cvat_work')
ref_dir = cvat_dir / 'ref_labels'
ref_dir.mkdir(parents=True, exist_ok=True)

random.seed(7)
train_imgs = sorted((DATASET_DIR / 'images' / 'train').glob('*.jpg'))
# 라벨이 1개 이상 있는 이미지 위주로 선별 (빈 이미지 라벨링은 학습 효과가 낮음)
cand = [p for p in train_imgs
        if (DATASET_DIR / 'labels' / 'train' / f'{p.stem}.txt').read_text().strip()]
cvat_set = random.sample(cand, min(CVAT_N, len(cand)))

with zipfile.ZipFile('/content/cvat_upload.zip', 'w') as z:
    for p in cvat_set:
        z.write(p, arcname=p.name)
        shutil.copy(DATASET_DIR / 'labels' / 'train' / f'{p.stem}.txt',
                    ref_dir / f'{p.stem}.txt')

print(f'✅ cvat_upload.zip 생성: {len(cvat_set)}장 / 참조 라벨 백업: {ref_dir}')
from google.colab import files
files.download('/content/cvat_upload.zip')   # 내 PC로 다운로드 → CVAT에 업로드

### 3-2. CVAT에서 라벨링하기 (Colab 밖 — 브라우저에서 진행)

**app.cvat.ai** (호스팅 버전, 무료)를 사용합니다. Docker 설치가 필요한 로컬 버전은 수업에서 다루지 않습니다.

**① 프로젝트·라벨 생성**
1. 로그인 → `Projects` → `+` → 이름 `vehicles5`
2. **Add label**로 라벨 5개 등록 — 철자를 정확히 (순서는 무관: 노트북 파서가 이름 기준으로 재매핑합니다 ✅):
   `ambulance` `bus` `car` `motorcycle` `truck`

**② 태스크 생성·이미지 업로드**
3. 프로젝트 안에서 `+ Create new task` → 이름 `batch01` → **Select files**에 `cvat_upload.zip` 업로드 → Submit

**③ 라벨링 (핵심 작업)**
4. 생성된 Job 열기 → 왼쪽 도구에서 **Rectangle** 선택 → 클래스 지정 후 드래그로 박스
5. 단축키: `N` 새 박스 · `Ctrl+S` 저장(수시로!) · `F`/`D` 다음/이전 프레임

**라벨링 규칙 (참조 라벨과 같은 기준 — 채점에 직접 영향)**
- 보이는 부분만 **타이트하게** (여백 금지 — IoU 점수가 깎이는 1순위 원인)
- **가려진 차량**: 30% 이상 보이면 라벨링, 보이는 영역 기준으로 박스
- **잘린 차량**(화면 경계): 보이는 부분만
- 미니밴·SUV → `car` / 픽업트럭·트럭 → `truck` / 스쿠터 → `motorcycle` — 애매하면 팀 내 기준을 **먼저 합의**
  (이 "합의" 과정이 실무의 라벨링 가이드라인 문서가 되는 과정입니다)

**④ Export**
6. Job 저장 후 태스크 메뉴 → `Export task dataset` → 포맷 **`YOLO 1.1`** → zip 다운로드

⏱️ **수업 운영 팁 (강사)**: 라벨링 30분은 GPU가 노는 시간이 아닙니다 — 4장(1단계 학습)을 **원본 라벨로 먼저 돌려두고**,
라벨링이 끝나면 3-4~3-6을 실행해 병합본으로 2단계 학습을 돌리는 병렬 진행이 가능합니다.
CVAT를 건너뛰는 경우(시간 부족/계정 문제)에도 3-4~3-5만 생략하면 원본 라벨로 실습이 그대로 이어집니다.

### 3-3. CVAT Export 업로드 및 파싱 (✅ 검증 완료)

CVAT `YOLO 1.1` export zip 구조: `obj.names`(등록한 라벨명 목록) + `obj_train_data/*.txt`(YOLO 라인).
**함정**: txt 안의 클래스 번호는 여러분이 CVAT에 라벨을 **등록한 순서** 기준입니다. 표준 순서와 다르면
클래스가 통째로 뒤바뀐 채 학습됩니다 — 아래 파서가 `obj.names`를 읽어 이름 기준으로 재매핑합니다.
*(사전 검증: 등록 순서를 뒤섞은 모의 export에서 정확 재매핑 확인)*

In [ ]:
# CVAT export zip 업로드
from google.colab import files
up = files.upload()                      # YOLO 1.1 export zip 선택
cvat_zip = '/content/' + list(up.keys())[0]

# ===== 검증 완료 파서 =====
import zipfile, os

def parse_cvat_yolo11(zip_path, out_dir, canonical=CANONICAL):
    """CVAT 'YOLO 1.1' export → 표준 클래스 순서로 재매핑된 라벨 저장"""
    warns = []
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        names_member = next((n for n in z.namelist() if n.endswith('obj.names')), None)
        assert names_member, 'obj.names 없음 — Export 포맷이 YOLO 1.1인지 확인'
        cvat_names = [l.strip().lower() for l in
                      z.read(names_member).decode('utf-8').splitlines() if l.strip()]
        remap = {}
        for i, nme in enumerate(cvat_names):
            if nme in canonical:
                remap[i] = canonical.index(nme)
            else:
                warns.append(f"CVAT 라벨 '{nme}'은 표준 클래스에 없음 — 해당 박스 제외")
        count = 0
        for member in z.namelist():
            if not member.endswith('.txt') or member.endswith('train.txt'):
                continue
            lines_out = []
            for line in z.read(member).decode('utf-8').splitlines():
                p = line.split()
                if len(p) >= 5 and int(p[0]) in remap:
                    lines_out.append(' '.join([str(remap[int(p[0])])] + p[1:5]))
            with open(os.path.join(out_dir, os.path.basename(member)), 'w') as f:
                f.write('\n'.join(lines_out))
            count += 1
    return count, remap, warns

stu_dir = str(cvat_dir / 'student_labels')
n, remap, warns = parse_cvat_yolo11(cvat_zip, stu_dir)
print(f'✅ 파싱 완료: {n}개 라벨 파일 | 재매핑 {remap}')
for w in warns:
    print('⚠️', w)
assert n > 0, '라벨 파일이 없습니다 — Export 시 "Save images" 옵션 없이 annotations만 내보내도 됩니다'

### 3-4. ★내 라벨 채점★ — 참조 라벨과 IoU 대조 (✅ 검증 완료)

여러분의 40장 라벨을 Open Images 참조 라벨과 그리디 IoU 매칭으로 대조합니다. 세 가지 실패 유형이 분리 집계됩니다:

| 유형 | 의미 | 흔한 원인 |
|---|---|---|
| **미검출**(missed) | 참조엔 있는데 내가 안 그림 | 작은 차량, 가려진 차량 무시 |
| **과검출**(extra) | 내가 그렸는데 참조엔 없음 | 30% 미만 노출 차량, 배경 오인 |
| **클래스 오류**(cls_err) | 위치는 맞는데 이름이 다름 | 미니밴↔truck, 스쿠터↔motorcycle 경계 |

> 💬 **강의 포인트**: 점수가 낮게 나온 학생의 사례를 화면에 띄워 함께 보세요.
> 대부분 "학생이 틀린 것"이 아니라 **참조 라벨도 애매한 경우**입니다 — Open Images 라벨에도 노이즈가 있습니다.
> "정답 라벨이라는 것도 결국 누군가의 판단"이라는 깨달음이 이 실습의 진짜 수확입니다.

In [ ]:
# ===== 검증 완료 채점기 =====
import numpy as np

def yolo_to_xyxy(row):
    c, cx, cy, w, h = row
    return int(c), np.array([cx-w/2, cy-h/2, cx+w/2, cy+h/2])

def iou(a, b):
    x1, y1 = np.maximum(a[:2], b[:2]); x2, y2 = np.minimum(a[2:], b[2:])
    inter = max(x2-x1, 0) * max(y2-y1, 0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def read_yolo(path):
    rows = []
    if os.path.exists(path):
        for line in open(path).read().splitlines():
            p = line.split()
            if len(p) >= 5:
                rows.append(tuple(map(float, p[:5])))
    return rows

tot = dict(matched=0, cls_ok=0, missed=0, extra=0)
ious, per_img = [], []
for rp in sorted(ref_dir.glob('*.txt')):
    S = [yolo_to_xyxy(r) for r in read_yolo(f'{stu_dir}/{rp.name}')]
    R = [yolo_to_xyxy(r) for r in read_yolo(str(rp))]
    used = set(); m = cls_ok = 0
    for rc, rb in R:
        best, bi = 0.0, -1
        for si, (sc, sb) in enumerate(S):
            if si not in used:
                v = iou(rb, sb)
                if v > best:
                    best, bi = v, si
        if best >= 0.5 and bi >= 0:
            used.add(bi); m += 1; ious.append(best)
            cls_ok += int(S[bi][0] == rc)
    tot['matched'] += m; tot['cls_ok'] += cls_ok
    tot['missed'] += len(R) - m; tot['extra'] += len(S) - m
    per_img.append((rp.stem, len(R) - m, len(S) - m, m - cls_ok))

print(f"매칭 {tot['matched']} | 클래스 일치 {tot['cls_ok']} | "
      f"미검출 {tot['missed']} | 과검출 {tot['extra']}")
print(f"매칭 박스 평균 IoU: {np.mean(ious):.3f}" if ious else '매칭 없음')
worst = sorted(per_img, key=lambda r: -(r[1]+r[2]+r[3]))[:5]
print('\n확인해볼 이미지 TOP5 (미검출/과검출/클래스오류):')
for s, a, b, c in worst:
    print(f'  {s}: {a}/{b}/{c}')
# 📊 초심자 전형: 평균 IoU 0.75~0.88, 미검출이 과검출보다 많음 (작은 차량 놓침)

### 3-5. 라벨 병합 및 GT 시각화

채점을 마친 **내 라벨을 실제 train 라벨로 채택**해 학습에 반영합니다 (원본 유지를 원하면 `ADOPT=False`).
이후 5클래스 GT를 눈으로 최종 확인합니다 — 여기서 박스가 어긋나 보이면 앞 단계(재매핑·좌표)를 의심하세요.

In [ ]:
ADOPT_STUDENT_LABELS = True    # False면 원본(Open Images) 라벨 그대로 학습

if ADOPT_STUDENT_LABELS:
    replaced = 0
    for sp in Path(stu_dir).glob('*.txt'):
        dst = DATASET_DIR / 'labels' / 'train' / sp.name
        if dst.exists():
            shutil.copy(sp, dst); replaced += 1
    print(f'✅ 학생 라벨 채택: {replaced}장 교체 (나머지 {len(cand)-replaced}장은 원본 라벨)')
else:
    print('원본 라벨 유지')

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

CLASS_NAMES = CANONICAL
COLORS = [(220, 20, 60), (30, 144, 255), (50, 205, 50), (255, 165, 0), (148, 0, 211)]

samples = random.sample(sorted((DATASET_DIR / 'images' / 'train').glob('*.jpg')), 8)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    lbl_path = DATASET_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            if not line.strip():
                continue
            c, cx, cy, w, h = map(float, line.split())
            c = int(c)
            x1, y1 = int((cx - w/2) * W), int((cy - h/2) * H)
            x2, y2 = int((cx + w/2) * W), int((cy + h/2) * H)
            cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[c], 3)
            cv2.putText(img, CLASS_NAMES[c], (x1, max(y1-8, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, COLORS[c], 2)
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 3-6. 데이터셋 설정 파일(yaml) 생성

yolov9-dpi의 `data/` 아래에 학습 설정 파일을 만듭니다.

In [ ]:
%%writefile data/vehicles5.yaml
train: /content/datasets/vehicles/images/train
val: /content/datasets/vehicles/images/val
test: /content/datasets/vehicles/images/test

nc: 5
names: ['ambulance', 'bus', 'car', 'motorcycle', 'truck']

### 3-7. Ultralytics 폰트 사전 배치

학습 시작 시 `check_dataset()`이 라벨 표시용 `Arial.ttf`를 `ultralytics.com/assets`에서 내려받으려 하는데,
이 주소는 현재 **HTTP 308 리다이렉트**를 반환하고 Python 3.8의 urllib은 308을 따라가지 못해 학습이 시작조차 안 됩니다.

`check_font()`는 파일이 이미 존재하면 다운로드를 건너뛰므로, GitHub의 새 주소에서 폰트를 **미리 받아 배치**해 우회합니다.

In [ ]:
import os, urllib.request

font_dir = os.path.expanduser('~/.config/Ultralytics')
os.makedirs(font_dir, exist_ok=True)

base = 'https://github.com/ultralytics/assets/releases/download/v0.0.0'
for fname in ['Arial.ttf', 'Arial.Unicode.ttf']:
    dst = os.path.join(font_dir, fname)
    if not os.path.exists(dst):
        urllib.request.urlretrieve(f'{base}/{fname}', dst)
        print(f'배치 완료: {dst} ({os.path.getsize(dst)/1e3:.0f} KB)')
    else:
        print(f'이미 존재: {dst}')

---
## 4. 1단계 학습: 기본 모델 (bsnet-t)

`bsnet-t.yaml` 설정으로 기본 모델을 처음부터(`--weights ''`) 학습합니다.

⏱️ T4 기준 약 20~30분 소요. 진행 중 epoch별 `mAP50` 이 점점 올라가는지 확인하세요.

⚠️ **재실행 주의**: 학습을 다시 돌리면 `runs/train/bsnet-t2`, `bsnet-t3`처럼 새 폴더가 생깁니다.
이 경우 이후 컴파일 단계의 경로도 함께 바꿔야 하므로, 재학습 시에는 기존 폴더를 지우고 시작하는 것을 권장합니다.

In [ ]:
!conda run --no-capture-output -n py38 python3 train.py \
  --workers 16 \
  --device 0 \
  --batch 32 \
  --data data/vehicles5.yaml \
  --img 416 \
  --cfg models/deeper-i/bsnet-t.yaml \
  --weights '' \
  --name bsnet-t \
  --hyp hyp.scratch.yaml \
  --min-items 0 \
  --epochs 20 \
  --close-mosaic 3 \
  --optimizer SGD

### 4-1. 1단계 학습 결과 확인

In [ ]:
from IPython.display import Image as IPImage, display
from pathlib import Path

run_dir = Path('runs/train/bsnet-t')
for name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = run_dir / name
    if p.exists():
        print(f'--- {name} ---')
        display(IPImage(str(p), width=900))
    else:
        print(f'{name} 없음 (학습 폴더 이름 확인: runs/train/ 아래를 확인하세요)')

---
## 5. 2단계 학습: 최적화 모델 (bsnet-t-o)

1단계의 `best.pt`를 초기 가중치로 사용해 NPU 배포용 최적화 모델을 학습합니다.
하이퍼파라미터도 `hyp.optimize.yaml`로 변경됩니다.

⏱️ T4 기준 약 20~30분 소요.

In [ ]:
!conda run --no-capture-output -n py38 python3 train.py \
  --workers 16 \
  --device 0 \
  --batch 32 \
  --data data/vehicles5.yaml \
  --img 416 \
  --cfg models/deeper-i/bsnet-t-o.yaml \
  --weights './runs/train/bsnet-t/weights/best.pt' \
  --name bsnet-t-o \
  --hyp hyp.optimize.yaml \
  --min-items 0 \
  --epochs 20 \
  --close-mosaic 3 \
  --optimizer SGD

In [ ]:
# 두 단계 가중치가 모두 생성되었는지 확인
!ls -lh runs/train/bsnet-t/weights/best.pt runs/train/bsnet-t-o/weights/best.pt

---
## 6. TACHY 컴파일용 optional 파일 업로드

`fc.out`, `block_4bit.out` 두 파일은 저장소에 포함되어 있지 않습니다.
yolov9-dpi README에 안내된 경로에서 **내 PC로 다운로드해 둔 뒤**, 아래 셀을 실행하면 나오는 업로드 버튼으로 **두 파일을 한 번에 선택**해 올립니다.

(파일 크기에 따라 업로드에 수 분이 걸릴 수 있습니다. 셀 좌측 진행 표시가 끝날 때까지 기다리세요.)

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('compile/optional', exist_ok=True)
uploaded = files.upload()  # fc.out, block_4bit.out 두 파일 선택

for name in uploaded:
    shutil.move(name, f'compile/optional/{name}')
    print('배치 완료:', f'compile/optional/{name}')

!chmod +x ./compile/optional/fc.out ./compile/optional/block_4bit.out
!ls -lh ./compile/optional/

---
## 7. 컴파일 설정 확인

`compile/compile_linux.py` 상단의 사용자 설정 영역을 확인합니다.

| 설정 | 이번 실습 값 | 설명 |
| --- | --- | --- |
| `PRE_PARAM_DIR` | `../runs/train/bsnet-t/weights` | 기본 모델 가중치 폴더 |
| `OPT_PARAM_DIR` | `../runs/train/bsnet-t-o/weights` | 최적화 모델 가중치 폴더 |
| `ONNX_INPUT_SHAPE` | `1 3 256 416` | (B, C, H, W) |
| `TACHY_INPUT_SHAPE` | `1 256 416 3` | (B, H, W, C) |

💡 **왜 416×416이 아니라 256×416인가?**

학습은 `--img 416`(정사각 기준)으로 했지만, 모델은 fully convolutional이라 다른 입력 크기로도 추론이 가능합니다.
라즈베리파이의 카메라 예제(`app.py`)와 후처리 설정(`post_process_256x416.json`)이
**256×416 (가로형)** 기준으로 구성되어 있으므로, 카메라 시연을 위해 저장소 기본값인 256×416을 그대로 사용합니다.
이름(`--name`)도 매뉴얼 기본값(`bsnet-t`, `bsnet-t-o`)을 그대로 썼기 때문에 **수정 없이 기본 설정으로 컴파일 가능합니다.**

아래 셀로 현재 설정값을 출력해 위 표와 일치하는지 확인만 하세요.

⚠️ **학습 폴더 이름이 밀린 경우** (`runs/train/` 아래가 `bsnet-t2`, `bsnet-t-o2` 등):
이전 학습이 중간에 실패하면 빈 폴더가 남아 재학습 결과가 `-2` 접미사 폴더로 들어갑니다.
이때는 아래 두 가지를 확인·수정해야 합니다.

1. **2단계 학습의 `--weights` 경로**가 실제 가중치가 있는 폴더(`bsnet-t2` 등)를 가리켰는지
2. `compile/compile_linux.py`의 `PRE_PARAM_DIR`/`OPT_PARAM_DIR`를 실제 폴더명으로 수정

```python
# 실제 가중치 존재 확인 (두 파일 모두 수 MB 이상이어야 정상)
!ls -lh runs/train/*/weights/best.pt
```

In [ ]:
!sed -n '1,25p' compile/compile_linux.py

---
## 8. TACHY 컴파일 실행

컴파일 스크립트는 내부적으로 다음을 수행합니다.

1. 두 모델의 `best.pt` → 배포용 PyTorch 파일 변환
2. PyTorch → ONNX 변환
3. ONNX → TACHY layer 형식 변환
4. TACHY block 형식 변환 (4bit 양자화 포함)
5. 최종 `.tachyrt` runtime 파일 생성

실행하면 저장소 루트에 `YYMMDD_HHMMSS/` 형태의 결과 폴더가 생성됩니다. **출력 로그에 찍히는 폴더명을 기록해 두세요.**

In [ ]:
%cd compile
!conda run --no-capture-output -n py38 python ./compile_linux.py
%cd ..

### 8-1. 산출물 확인 — 아래 `OUTPUT_DIR`을 컴파일 로그에 출력된 폴더명으로 수정 후 실행

In [ ]:
# ⚠️ 컴파일 로그에 출력된 실제 폴더명으로 수정하세요 (예: 260822_153000)
OUTPUT_DIR = '260822_153000'

import glob
files = glob.glob(f'{OUTPUT_DIR}/model_*.tachyrt') + glob.glob(f'compile/{OUTPUT_DIR}/model_*.tachyrt')
if files:
    for f in files:
        print('✅ 컴파일 산출물:', f)
else:
    print('❌ .tachyrt 파일을 찾지 못했습니다. OUTPUT_DIR 경로를 확인하세요.')
    !ls -d 2*_*/ compile/2*_*/ 2>/dev/null

### 8-2. 산출물 백업 다운로드 (필수)

세션이 끊기면 로컬 디스크의 결과물이 모두 사라집니다.
학습 가중치(`best.pt` 2개)와 컴파일 산출물(`.tachyrt`)을 zip으로 묶어 내 PC로 내려받습니다.
위 8-1 셀에서 설정한 `OUTPUT_DIR` 값을 그대로 사용합니다.

In [ ]:
import shutil, glob, os
from google.colab import files

BACKUP_DIR = '/content/backup/vehicles5'
os.makedirs(BACKUP_DIR, exist_ok=True)

targets = [
    'runs/train/bsnet-t/weights/best.pt',
    'runs/train/bsnet-t-o/weights/best.pt',
]
targets += glob.glob(f'{OUTPUT_DIR}/model_*.tachyrt')
targets += glob.glob(f'compile/{OUTPUT_DIR}/model_*.tachyrt')

for t in targets:
    if os.path.exists(t):
        dst = os.path.join(BACKUP_DIR, t.replace('/', '__'))
        shutil.copy(t, dst)
        print('백업:', t)
    else:
        print('⚠️ 없음 (경로 확인):', t)

zip_path = shutil.make_archive('/content/vehicles5_artifacts', 'zip', BACKUP_DIR)
print('압축 완료:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
files.download(zip_path)

---
## 9. 모니터 시연용 테스트 슬라이드 생성

학습에 사용하지 않은 **test 분할**에서 클래스별 3장씩 골라, 전체화면 슬라이드쇼 HTML을 만듭니다.
이 파일을 강의실 모니터에 띄우고 카메라(imx219)로 촬영하며 실시간 탐지를 시연합니다.

**슬라이드 조작법**
- `←` / `→` : 이전/다음 이미지
- `C` : 정답 클래스명 표시/숨김 (기본 숨김 — 모델이 먼저 맞히게 한 뒤 정답 공개)
- `F11` : 브라우저 전체화면

In [ ]:
import base64, random
from pathlib import Path

CLASS_NAMES = ['ambulance', 'bus', 'car', 'motorcycle', 'truck']
PER_CLASS = 3
random.seed(42)

# 단일 클래스만 포함된 test 이미지를 클래스별로 선별 (시연 판독이 명확하도록)
picks = {i: [] for i in range(5)}
for lp in sorted((DATASET_DIR / 'labels' / 'test').glob('*.txt')):
    lines = [l.split() for l in lp.read_text().splitlines() if l.strip()]
    cls_set = {int(l[0]) for l in lines}
    if len(cls_set) == 1:
        c = cls_set.pop()
        if len(picks[c]) < PER_CLASS:
            img = DATASET_DIR / 'images' / 'test' / (lp.stem + '.jpg')
            if img.exists():
                picks[c].append(img)

# 희귀 클래스는 단일-클래스 이미지가 부족할 수 있음 → 해당 클래스 포함 이미지로 폴백
for c_id, ps in picks.items():
    if len(ps) < PER_CLASS:
        for lp in sorted((DATASET_DIR / 'labels' / 'test').glob('*.txt')):
            lines = [l.split() for l in lp.read_text().splitlines() if l.strip()]
            if any(int(l[0]) == c_id for l in lines):
                img = DATASET_DIR / 'images' / 'test' / (lp.stem + '.jpg')
                if img.exists() and img not in ps:
                    ps.append(img)
            if len(ps) >= PER_CLASS:
                break

slides = [(CLASS_NAMES[c], p) for c, ps in picks.items() for p in ps]
random.shuffle(slides)
print(f'선별된 슬라이드: {len(slides)}장')

slide_html = ''
for i, (name, p) in enumerate(slides):
    b64 = base64.b64encode(p.read_bytes()).decode()
    slide_html += (f'<div class="slide" data-answer="{name}">'
                   f'<img src="data:image/jpeg;base64,{b64}"></div>')

html = f'''<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Vehicles Demo</title>
<style>
  body {{ margin:0; background:#000; overflow:hidden; }}
  .slide {{ display:none; width:100vw; height:100vh;
           align-items:center; justify-content:center; }}
  .slide.active {{ display:flex; }}
  .slide img {{ max-width:100vw; max-height:100vh; object-fit:contain; }}
  #answer {{ position:fixed; top:24px; left:50%; transform:translateX(-50%);
            color:#fff; background:rgba(0,0,0,.65); padding:10px 28px;
            font:bold 42px sans-serif; border-radius:12px; display:none; }}
  #counter {{ position:fixed; bottom:16px; right:24px; color:#666;
             font:16px monospace; }}
</style></head><body>
{slide_html}
<div id="answer"></div><div id="counter"></div>
<script>
  const slides = document.querySelectorAll('.slide');
  const answer = document.getElementById('answer');
  const counter = document.getElementById('counter');
  let idx = 0, showAns = false;
  function render() {{
    slides.forEach((s, i) => s.classList.toggle('active', i === idx));
    answer.textContent = slides[idx].dataset.answer;
    answer.style.display = showAns ? 'block' : 'none';
    counter.textContent = (idx + 1) + ' / ' + slides.length;
  }}
  document.addEventListener('keydown', e => {{
    if (e.key === 'ArrowRight') idx = (idx + 1) % slides.length;
    else if (e.key === 'ArrowLeft') idx = (idx - 1 + slides.length) % slides.length;
    else if (e.key.toLowerCase() === 'c') showAns = !showAns;
    render();
  }});
  render();
</script></body></html>'''

out_path = Path('/content/yolov9-dpi/monitor_demo.html')
out_path.write_text(html, encoding='utf-8')
print('저장 완료:', out_path)

from google.colab import files
files.download(str(out_path))

---
## 10. 라즈베리파이 배포 및 시연 (Colab 밖에서 진행)

이 단계부터는 **RPi4 + Tachy-Shield** 장비에서 진행합니다. (상세 절차: `IPA_RPI4_TACHY_SHIELD_MANUAL.md` 참조)

보드의 예제 프로젝트는 `~/vehicles5/` 폴더에 있고, 실행 진입점은 `app.py`입니다.

```text
~/vehicles5/
├── app.py                          ← 실행 진입점 (수정본으로 교체)
├── configs/
│   └── vehicles5_yolov9/
│       └── post_process_256x416.json   ← 4클래스 수정본 배치
├── params/
│   └── vehicles5_yolov9/
│       └── model_256x416x3_inv-f.tachyrt   ← 컴파일 산출물 배치
└── src/  (detection_utils.py 등)
```

### 10-1. 파일 배치

컴파일 산출물(`.tachyrt`)과 수정된 두 파일(`app.py`, `post_process_256x416.json`)을 PC에서 보드로 복사합니다.

```bash
# PC에서 실행 (IP는 실제 장비 주소로)
ssh pi@<RPI_IP> 'mkdir -p ~/vehicles5/params/vehicles5_yolov9 ~/vehicles5/configs/vehicles5_yolov9'

scp model_256x416x3_inv-f.tachyrt \
    pi@<RPI_IP>:~/vehicles5/params/vehicles5_yolov9/
scp post_process_256x416.json \
    pi@<RPI_IP>:~/vehicles5/configs/vehicles5_yolov9/
scp app.py pi@<RPI_IP>:~/vehicles5/
```

⚠️ `.tachyrt` 파일명이 `inv-f`가 아니라면(`inv-t` 등) `app.py`의 `model_path` 줄을 실제 파일명으로 맞추세요.

### 10-2. 수정 파일 내용 (참고)

함께 배포되는 두 파일에는 다음 수정이 이미 반영되어 있습니다. 직접 수정할 경우 참고하세요.

**`configs/vehicles5_yolov9/post_process_256x416.json`** — 2곳

```json
{
    "SHAPES_INPUT": [256, 416, 3],
    "SHAPES_OUTPUT": [[32,52,9],[16,26,9],[8,13,9]],
    "OBJ_THRESHOLD":0.2,
    "NMS_THRESHOLD":0.2,
    "PRE_THRESHOLD":0.2,
    "N_MAX_OBJ": 30,
    "N_CLASSES":5
}
```

- `N_CLASSES`: 1 → **5**
- `SHAPES_OUTPUT` 마지막 차원: 5 → **9** — 후처리(`post_process.py`)가 채널을 `4(박스) + N_CLASSES`로 분해하므로 4+5=9

**`app.py`** — 3곳

1. import 아래에 클래스 이름 추가:

```python
CLASS_NAMES = ["ambulance", "bus", "car", "motorcycle", "truck"]
```

2. `main()`의 모델·설정 경로 교체:

```python
    model_path = (
        base_dir
        / "params/vehicles5_yolov9/model_256x416x3_inv-f.tachyrt"
    )
    post_process_config_path = (
        base_dir
        / "configs/vehicles5_yolov9/post_process_256x416.json"
    )
```

3. `_draw_detections()`의 라벨을 클래스 번호 대신 차량 이름으로:

```python
            name = (
                CLASS_NAMES[class_id]
                if 0 <= class_id < len(CLASS_NAMES)
                else str(class_id)
            )
            label = "{} {:.3f}".format(name, score)
```

### 10-3. 실행 순서 (표준 진단 시퀀스)

```bash
# 0) Tachy-Shield 부팅 완료 확인 후 venv 활성화
source /opt/venv/bin/activate
cd ~/vehicles5

# 1) 디바이스 상태 확인
python3 get_device_status.py

# 2) 성능 측정 (선택 — 새 모델의 latency/FPS 기록용)
./check_latency.sh
./check_throughput.sh

# 3) 실시간 객체 탐지 실행 (q 또는 ESC로 종료)
python3 app.py
```

화면 창에 카메라 영상과 함께 `bus 0.87` 형태의 탐지 라벨이 표시되면 성공입니다.

### 10-4. 모니터 재촬영 시연 팁

| 항목 | 권장 설정 | 이유 |
| --- | --- | --- |
| 카메라~모니터 거리 | 50cm ~ 1m | 모아레(픽셀 격자 간섭) 최소화 |
| 모니터 밝기 | 중간 이하 + 실내조명 ON | 학습 데이터(실사)와 노출 분포 유사화 |
| 첫 시연 클래스 | **bus** | 크고 형태가 뚜렷해 재촬영 열화에 가장 강건 |
| 정답 공개 | 모델 탐지 후 `C` 키 | 수강생이 모델 판단을 먼저 보게 하는 연출 |
| 미검출이 잦을 때 | json의 `OBJ_THRESHOLD` 0.2 → 0.15 | 재촬영으로 낮아진 신뢰도 보정 |

💡 **수업 연결 포인트**: 모니터 재촬영 시 신뢰도(confidence)가 원본 대비 떨어지는 현상은
"학습 분포와 추론 분포가 다르면 성능이 저하된다"는 **도메인 시프트/모델 드리프트** 개념의 살아있는 예시입니다.
latency ~130ms인데 ~30 FPS가 나오는 이유(듀얼코어 NPU 파이프라이닝)와 함께 토론 주제로 활용하세요.

---

## 실습 완료 체크리스트

- [ ] 1단계/2단계 학습 완료, `best.pt` 2개 확인
- [ ] `.tachyrt` 컴파일 산출물 확인 및 백업 다운로드 (8-2)
- [ ] `monitor_demo.html` 다운로드
- [ ] CVAT 라벨 채점 결과 기록 (평균 IoU, 미검출/과검출/클래스오류 수 — 리포트 제출용)
- [ ] 보드 `~/vehicles5/`에 3개 파일 배치 (`.tachyrt`, json, `app.py`)
- [ ] `python3 app.py` 실행, 차량 이름 라벨 표시 확인
- [ ] 모니터 재촬영 시연 성공 (bus → 나머지 클래스 순)